In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Paths come from path_config.yaml; edit that file's current_workstation to switch machines.
from path_config import PMOF_CODE_DIR, DATA_BASE_DIR as data_base_dir

# Add project root to sys.path (so imports like src.data work)
if str(PMOF_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(PMOF_CODE_DIR))

from src import logger

from ultralytics import YOLO

import random
from src.visualization import results_to_frames, save_video, VIZ_PARAMS
from src.data import imgid_to_imgpath, imgid_to_annpath, recid_to_annpath
from src.data import recordid_to_imageids, read_annotation

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import shutil

from src.data import list_record_ids
record_ids = list_record_ids()
from src.visualization import VIZ_PARAMS
bbox_colors = VIZ_PARAMS['gt_bbox_colors']

from src.data import imgid_to_imgpath, imgid_to_annpath, recid_to_annpath, recordid_to_imageids, read_annotation, list_record_ids

In [22]:
# load model
def get_obb_alert(model_path, frame):
    #load model
    #get data
    #post-process prediction
    
    model = YOLO(model_path)  # load a custom model
    
    # Predict with the model
    results = model(frame)  # predict on an image

    other_actions_counter = 0
    
    # Access the results
    for result in results:
        xywhr = result.obb.xywhr  # center-x, center-y, width, height, angle (radians)
        xyxyxyxy = result.obb.xyxyxyxy  # polygon format with 4-points
        names = [result.names[cls.item()] for cls in result.obb.cls.int()]  # class name of each box
        confs = result.obb.conf  # confidence score of each box
        if names != 'seated':
            other_actions_counter += 1
    return 1 if other_actions_counter != 0 else 0
    

def get_cls_alert(model_path, frame):
    #load model
    #get data
    #post-process prediction   

    # Load a model
    model = YOLO(model_path)  # load a custom model

    # Predict with the model
    results = model(frame)  # predict on an image

    # Access the results
    for result in results:
        top1 = result.probs.top1  # top predicted class ID
        top1_conf = result.probs.top1conf  # top prediction confidence
        top1_name = result.names[top1]  # top predicted class name
    return top1

def get_gt_alert(img_id):
    #get annotation path
    #load annotation
    #process annotation

    # Get annotation paths
    annpath = imgid_to_annpath(img_id)
    anns = read_annotation(annpath, img_id)

    # Only person annotations matter for action classification
    person_anns = [
        ann for ann in anns
        if ann.category_name == "person"
    ]

    # Any non-seated action -> "other"
    has_other_action = any(
        box.action != 'seated'
        for box in person_anns
    )
    gt_alert = 1 if has_other_action else 0
    return gt_alert
    
def eval_alert(img_id, model_type, model_path):
    # need to define tn, tp, fp, fn
    #load frame
    frame = Path(imgid_to_imgpath(img_id))
    #frame = np.load(img_path)
    #load annotation and get gt_alter
    gt_alert = get_gt_alert(img_id)
    if model_type == 'obb':
        pred_alert = get_obb_alert(model_path, frame)
    if model_type == 'cls':
        pred_alert = get_cls_alert(model_path, frame)

    #analyse fp, fn, tn, tp
    if gt_alert == 0 and pred_alert == 0:
        return 'tn'
    if gt_alert == 1 and pred_alert == 1:
        return 'tp'
    if gt_alert == 0 and pred_alert == 1:
        return 'fp'
    if gt_alert == 1 and pred_alert == 0:
        return 'fn'

def compare_results(img_id, model_paths, results):
    for model_type in ['cls', 'obb']:
        result = eval_alert(img_id, model_type, model_paths[model_type])
        results[model_type][result].append(img_id)
    return results

In [30]:
val_record_ids = ["rec29", "rec30"]

cls_path = '/home/swermuth/pmof-symposium/runs/classify/train/weights/best.pt'
obb_path = '/home/swermuth/pmof_action/runs/obb/deletelater4/weights/best.pt'

results = {'cls': {'tn':[], 'tp':[], 'fp':[], 'fn':[]}, 'obb': {'tn':[], 'tp':[], 'fp':[], 'fn':[]}}
model_paths = {'cls' : cls_path, 'obb':obb_path}


for rec_id in val_record_ids:
    image_ids = recordid_to_imageids(rec_id)
test_imgids = recordid_to_imageids("rec30")
random.shuffle(test_imgids)

for img_id in test_imgids[:30]:
    results = compare_results(img_id, model_paths, results)


image 1/1 /home/swermuth/PMOF/images/rec30/rec30_001885.png: 320x320 seated 0.88, other 0.12, 3.9ms
Speed: 17.4ms preprocess, 3.9ms inference, 0.0ms postprocess per image at shape (1, 3, 320, 320)

image 1/1 /home/swermuth/PMOF/images/rec30/rec30_001885.png: 320x320 1 seated, 9.8ms
Speed: 0.8ms preprocess, 9.8ms inference, 0.3ms postprocess per image at shape (1, 3, 320, 320)

image 1/1 /home/swermuth/PMOF/images/rec30/rec30_001773.png: 320x320 other 0.96, seated 0.04, 3.7ms
Speed: 17.2ms preprocess, 3.7ms inference, 0.0ms postprocess per image at shape (1, 3, 320, 320)

image 1/1 /home/swermuth/PMOF/images/rec30/rec30_001773.png: 320x320 (no detections), 9.6ms
Speed: 0.7ms preprocess, 9.6ms inference, 0.2ms postprocess per image at shape (1, 3, 320, 320)

image 1/1 /home/swermuth/PMOF/images/rec30/rec30_001504.png: 320x320 seated 1.00, other 0.00, 3.6ms
Speed: 16.8ms preprocess, 3.6ms inference, 0.0ms postprocess per image at shape (1, 3, 320, 320)

image 1/1 /home/swermuth/PMOF/imag

In [41]:
for resulttype in results['cls'].keys():
    print(resulttype)
    print(len(results['cls'][resulttype]))

tn
13
tp
0
fp
11
fn
6


In [42]:
for resulttype in results['obb'].keys():
    print(resulttype)
    print(len(results['obb'][resulttype]))

tn
0
tp
6
fp
24
fn
0


In [ ]:
#check if ultrayltics validation gets the same confusion matrix as my code
#need to run on full validation set (or sample validation smaller)

#sanity check: do the numbers add up to the number of frames
#sanity check: there should be no overlap between the lists

#how about confidence threshold in my code

#combine with PMOF person detection in some way???

#improve models themself